# TP0 : DIMENSIONNEMENT ET PLANIFICATION CELLULAIRE — GSM (2G)
## Modèle à trois étages — Durée : 1h30

### Objectifs
- Maîtriser le modèle de propagation à 3 étages et le bilan de liaison
- Dimensionner une cellule (rayon de couverture)
- Choisir le motif cellulaire à partir du rapport C/I
- Dimensionner un réseau : couverture vs capacité (Erlang B)

### ⚠️ Instructions
- Complétez les zones marquées **`# TODO`**
- Exécutez chaque cellule avec **Shift+Entrée**, dans l'ordre
- Répondez aux questions en texte dans les cellules **Réponse**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log10, sqrt, pi, ceil
plt.rcParams['figure.figsize'] = (10, 5)
print("✅ Bibliothèques chargées")

## EXERCICE 1 : BILAN DE LIAISON

> **Pour comprendre.** Une station de base émet une puissance fixe ; en s'éloignant, le signal s'affaiblit d'abord comme en espace libre (en 1/r²), puis beaucoup plus vite dès que des bâtiments, le sol et la végétation s'en mêlent : l'exposant α passe de 2 à 3 ou 4, ce qui signifie que doubler la distance divise la puissance reçue par 8 à 16 au lieu de 4. Le bilan de liaison consiste à additionner, en décibels, tout ce qui aide (puissance émise, gains d'antennes) et à retrancher tout ce qui nuit (affaiblissement, pertes, marges), pour trouver la distance à laquelle le signal reçu passe sous la sensibilité du récepteur : c'est le rayon de couverture. Travailler en dB n'est pas une coquetterie : cela transforme les produits en sommes et permet de manipuler des rapports de 10¹² sans se tromper d'ordre de grandeur.


**Modèle à 3 étages :**

$$P_r = P_e\,G_e\,G_r\,k\,rac{\lambda^2}{r^{lpha}}$$

### Données
- f = 900 MHz ; P_e = 40 W ; G_e = 11 dBi ; G_r = 0 dBi ; α = 3,5 ; k = 1
- Sensibilité du récepteur : −102 dBm

In [ ]:
c = 3e8; f = 900e6
P_e_W = 40; G_e_dB = 11; G_r_dB = 0
alpha = 3.5; k = 1
print(f"f = {f/1e6:.0f} MHz, P_e = {P_e_W} W, alpha = {alpha}")

### Q1.1 — Longueur d'onde $\lambda = c / f$

In [ ]:
lambda_m = # TODO : COMPLÉTEZ ICI
print(f"λ = {lambda_m:.4f} m = {lambda_m*100:.1f} cm")

### Q1.2 — Puissance émise en dBm : $P_{dBm} = 10\log_{10}(P_W \times 1000)$

In [ ]:
P_e_dBm = # TODO : COMPLÉTEZ ICI
print(f"P_e = {P_e_dBm:.2f} dBm")

### Q1.3 — Gains en linéaire : $G = 10^{G_{dB}/10}$

In [ ]:
G_e_lin = # TODO : COMPLÉTEZ ICI
G_r_lin = # TODO : COMPLÉTEZ ICI
print(f"G_e = {G_e_lin:.2f}   G_r = {G_r_lin:.2f}")

### Q1.4 — Fonction puissance reçue (modèle à 3 étages)

In [ ]:
def puissance_recue(r_m, P_e, G_e, G_r, lambda_m, alpha, k=1):
    """Renvoie (P_r en W, P_r en dBm) à la distance r_m (m)."""
    P_r_W = # TODO : COMPLÉTEZ ICI
    return P_r_W, 10 * log10(P_r_W * 1000)

for r in (1000, 5000):
    P_W, P_dBm = puissance_recue(r, P_e_W, G_e_lin, G_r_lin, lambda_m, alpha, k)
    print(f"r = {r/1000:.0f} km : P_r = {P_W:.2e} W = {P_dBm:.2f} dBm")

**Q1.4 bis (réflexion)** — Pour $\alpha = 2$ (espace libre), la formule de Friis donne $P_r = P_e\,G_e\,G_r\left(\dfrac{\lambda}{4\pi r}\right)^2$.
Quelle valeur de k rend le modèle à 3 étages identique à Friis ? Que représente physiquement le fait de prendre k = 1 avec α = 3,5 ?

**Réponse :** _(à compléter)_

### Q1.6 — Rayon maximal : $R_{max} = \left(\dfrac{P_e\,G_e\,G_r\,k\,\lambda^2}{P_{sens}}\right)^{1/\alpha}$

In [ ]:
P_sens_dBm = -102
P_sens_W = 10 ** (P_sens_dBm / 10) / 1000
r_alpha = # TODO : COMPLÉTEZ ICI
R_max = # TODO : COMPLÉTEZ ICI
print(f"R_max = {R_max/1000:.2f} km")

### Graphique — puissance reçue en fonction de la distance

In [ ]:
d = np.linspace(100, 25000, 300)
P = [puissance_recue(x, P_e_W, G_e_lin, G_r_lin, lambda_m, alpha, k)[1] for x in d]
plt.plot(d/1000, P, lw=2, label='Puissance reçue')
plt.axhline(P_sens_dBm, color='r', ls='--', label='Sensibilité (−102 dBm)')
plt.axvline(R_max/1000, color='g', ls='--', label=f'R_max = {R_max/1000:.1f} km')
plt.xlabel('Distance (km)'); plt.ylabel('P_r (dBm)'); plt.grid(alpha=.3); plt.legend()
plt.title('Bilan de liaison — Exercice 1'); plt.show()

---
## EXERCICE 2 : INTERFÉRENCES CO-CANAL

> **Pour comprendre.** Le spectre est rare : un opérateur ne dispose que de quelques dizaines de MHz pour tout un pays. Il faut donc réutiliser les mêmes fréquences dans plusieurs cellules, suffisamment éloignées pour ne pas se brouiller. Le motif K est le nombre de cellules voisines qui se partagent la bande avant qu'une fréquence ne soit réutilisée : plus K est grand, plus les cellules co-canal sont loin (D = √(3K)·R) et meilleur est le rapport signal sur interférence C/I ; mais chaque cellule ne dispose alors que d'un K-ième de la bande. Le C/I se calcule en bordure de cellule, là où le mobile est le plus loin de sa station et le plus près des six cellules co-canal du premier anneau. Le planificateur cherche le plus petit K qui respecte le seuil de qualité, en gardant une marge pour les évanouissements que le modèle simple ne voit pas.


### Données
- P_e = 40 W ; G_e = G_r = 1 ; α = 4 ; R = 2 km
- Seuil C/I nominal : 9 dB (GSM)

Réseau hexagonal, 6 interféreurs du premier anneau à la distance $D = \sqrt{3K}\,R$.

In [ ]:
P_e2, G_e2, G_r2, alpha2, R = 40, 1, 1, 4, 2000
CI_seuil_dB = 9

### Q2.1 — Signal utile C en bordure de cellule (r = R)

In [ ]:
C, _ = puissance_recue(R, P_e2, G_e2, G_r2, lambda_m, alpha2, k)
print(f"C = {C:.2e} W")

### Q2.2 — Distance de réutilisation pour K = 3 : $D = \sqrt{3K}\,R$

In [ ]:
K = 3
D = # TODO : COMPLÉTEZ ICI
print(f"D = {D/1000:.2f} km")

### Q2.3 / Q2.4 — Interférence d'un co-canal, puis des 6 interféreurs

In [ ]:
I1, _ = puissance_recue(D, P_e2, G_e2, G_r2, lambda_m, alpha2, k)
I_total = # TODO : COMPLÉTEZ ICI
print(f"I1 = {I1:.2e} W   I_total = {I_total:.2e} W")

### Q2.5 — Rapport C/I

In [ ]:
CI_lin = # TODO : COMPLÉTEZ ICI
CI_dB = # TODO : COMPLÉTEZ ICI
print(f"C/I = {CI_lin:.1f} (lin) = {CI_dB:.2f} dB  → seuil {CI_seuil_dB} dB : {'✅ suffisant' if CI_dB >= CI_seuil_dB else '❌ insuffisant'}")

### Q2.6 — Valeur minimale de K

On montre facilement que $\dfrac{C}{I} = \dfrac{1}{6}\left(\sqrt{3K}\right)^{\alpha}$, indépendant de $P_e$ et de $R$.
Le tableau ci-dessous compare deux seuils : le seuil nominal (9 dB) et un seuil **avec marge** de 9 dB
pour absorber le shadowing et le fading (18 dB), ce qui correspond aux pratiques de planification.

In [ ]:
K_values = {1:[(1,0)], 3:[(1,1)], 4:[(2,0)], 7:[(2,1)], 9:[(3,0)], 12:[(2,2)], 13:[(3,1)]}
print(f"{'K':>3} {'(i,j)':>8} {'D (km)':>8} {'C/I (dB)':>9}  seuil 9 dB  seuil 18 dB")
for Kv, couples in K_values.items():
    Dk = sqrt(3*Kv) * R
    Ik, _ = puissance_recue(Dk, P_e2, G_e2, G_r2, lambda_m, alpha2, k)
    CIk = 10*log10(C / (6*Ik))
    print(f"{Kv:>3} {str(couples[0]):>8} {Dk/1000:>8.2f} {CIk:>9.2f}     {'✅' if CIk>=9 else '❌'}          {'✅' if CIk>=18 else '❌'}")

**Réponse Q2.6** — K_min au seuil nominal = ___ ; K_min avec marge = ___ .
Pourquoi les réseaux GSM réels utilisaient-ils des motifs 7 ou 9 plutôt que 3 ?

_(à compléter)_

### Q2.7 — Canaux par cellule : bande 25 MHz, espacement 200 kHz, motif K = 7

In [ ]:
bande_MHz, espacement_kHz, K_deploy = 25, 200, 7
nb_canaux_total = # TODO : COMPLÉTEZ ICI
canaux_par_cellule = # TODO : COMPLÉTEZ ICI
print(f"{nb_canaux_total} canaux au total → {canaux_par_cellule} canaux par cellule (K={K_deploy})")

---
## EXERCICE 3 : DIMENSIONNEMENT RÉSEAU

> **Pour comprendre.** Dimensionner un réseau, c'est répondre à deux questions indépendantes : combien de cellules pour couvrir la surface (question de propagation, exercice 1), et combien de cellules pour écouler le trafic (question de capacité, exercice 2 et table d'Erlang). Le trafic se mesure en Erlang : 1 Erlang = une ressource occupée en permanence ; un abonné qui téléphone 90 secondes par heure représente 25 mErlang. La loi d'Erlang B donne, pour un nombre de canaux, le trafic que l'on peut écouler avec un taux de blocage acceptable (2 %). En zone urbaine dense, c'est presque toujours la capacité qui impose le nombre de cellules : on obtient alors des cellules bien plus petites que ce que la portée radio permettrait, et le levier principal devient la sectorisation, qui triple le nombre de cellules par site sans construire de nouveaux pylônes.


### Données
- Surface S = 100 km² ; rayon de cellule R = 1 km (hexagone : S_cell = 2,6·R²)
- 100 000 abonnés ; 25 mErlang par abonné ; P_b = 2 %
- Bande 25 MHz, canaux de 200 kHz, motif K = 7, sites tri-sectorisés

In [ ]:
surface_km2, rayon_km = 100, 1
nb_abonnes, trafic_mErl, P_b = 100_000, 25, 0.02
# Table d'Erlang B à P_b = 2 % : canaux -> trafic admissible (Erlang)
erlang_B = {10:5.08, 15:9.01, 20:13.2, 25:17.5, 30:22.0, 35:26.7, 40:31.0, 45:35.9, 50:40.8, 55:45.7, 60:50.6}

def trafic_admissible(n_canaux):
    """Trafic admissible (Erlang) par interpolation linéaire dans la table."""
    xs, ys = list(erlang_B.keys()), list(erlang_B.values())
    return float(np.interp(n_canaux, xs, ys))

### Q3.2 — Trafic total offert

In [ ]:
trafic_total_Erl = # TODO : COMPLÉTEZ ICI
print(f"Trafic total = {trafic_total_Erl:,.0f} Erlang")

### Q3.3 — Trafic admissible par cellule
Avec K = 7, chaque cellule dispose de `canaux_par_cellule` canaux (Q2.7). Lisez dans la table d'Erlang B
le trafic qu'une cellule peut écouler à 2 % de blocage.

In [ ]:
trafic_par_cellule = # TODO : COMPLÉTEZ ICI
print(f"{canaux_par_cellule} canaux/cellule → {trafic_par_cellule:.1f} Erlang par cellule (P_b = 2 %)")

### Q3.4 — Nombre de cellules pour la COUVERTURE : $S_{cell} = 2{,}6\,R^2$

In [ ]:
surface_cellule = # TODO : COMPLÉTEZ ICI
nb_cellules_couverture = # TODO : COMPLÉTEZ ICI
print(f"S_cell = {surface_cellule:.2f} km² → {nb_cellules_couverture} cellules pour couvrir {surface_km2} km²")

### Q3.5 — Nombre de cellules pour la CAPACITÉ

In [ ]:
nb_cellules_capacite = # TODO : COMPLÉTEZ ICI
print(f"{trafic_total_Erl:.0f} Erl / {trafic_par_cellule:.1f} Erl par cellule → {nb_cellules_capacite} cellules")

### Q3.6 — Critère dimensionnant

In [ ]:
nb_cellules_deploy = max(nb_cellules_couverture, nb_cellules_capacite)
critere = 'CAPACITÉ' if nb_cellules_capacite > nb_cellules_couverture else 'COUVERTURE'
print(f"Couverture : {nb_cellules_couverture}   Capacité : {nb_cellules_capacite}   → critère dimensionnant : {critere}")
print(f"→ Déployer {nb_cellules_deploy} cellules")

**Réponse Q3.6** — Quel rayon de cellule effectif cela impose-t-il (R_eff = √(S / (2,6·N)) ) ? Que devient-il si le trafic par abonné double ?

_(à compléter)_

### Q3.7 — Sites tri-sectorisés (1 site = 3 cellules)

In [ ]:
nb_sites = # TODO : COMPLÉTEZ ICI
print(f"{nb_sites} sites tri-sectorisés")

### Q3.8 — Porteuses par cellule et bande réellement utilisée avec K = 7

In [ ]:
porteuses_par_cellule = canaux_par_cellule
bande_utilisee_MHz = # TODO : COMPLÉTEZ ICI
print(f"{porteuses_par_cellule} porteuses/cellule ; bande utilisée = {bande_utilisee_MHz:.1f} MHz sur {bande_MHz} MHz")

### Synthèse graphique

In [ ]:
cat = ['Couverture', 'Capacité', 'Déploiement']
val = [nb_cellules_couverture, nb_cellules_capacite, nb_cellules_deploy]
plt.bar(cat, val, color=['skyblue', 'coral', 'lightgreen'], edgecolor='k')
for i, v in enumerate(val): plt.text(i, v + 3, str(v), ha='center', fontweight='bold')
plt.ylabel('Nombre de cellules'); plt.title('Couverture vs capacité'); plt.grid(axis='y', alpha=.3); plt.show()

---
## 📝 SYNTHÈSE FINALE — à remplir
| | Résultat |
|---|---|
| λ | ___ cm |
| P_e | ___ dBm |
| R_max (Ex. 1) | ___ km |
| C/I pour K = 3 | ___ dB |
| K_min (9 dB / 18 dB) | ___ / ___ |
| Canaux par cellule (K = 7) | ___ |
| Critère dimensionnant | ___ |
| Cellules à déployer / sites | ___ / ___ |

**Conclusion (5 lignes) :** que limite en premier le réseau, la couverture ou la capacité ? Quel levier (rayon, motif, sectorisation, bande) permet d'y répondre ?